# SAM2 Point-Prompt Segmentation — Custom BBox (KSA)\n\nThis notebook downloads high-res satellite imagery for your **bounding box**, collects **point prompts** (map/GeoJSON/CSV), runs **SAM2** segmentation, cleans + vectorizes the result, and previews outputs.\n\n**Your bbox (lon/lat):** `[42.6768, 17.48022, 42.69809, 17.49266]`

In [ ]:
# Install deps (quiet)
!pip -q install leafmap samgeo geopandas shapely pyproj rioxarray rasterio folium

In [ ]:
import os, json, tempfile
import leafmap
import geopandas as gpd
from shapely.geometry import Polygon, box, Point
from samgeo.sam2 import SamGeo2
from samgeo import regularize

# -------------------------------
# User config (edit if you like)
# -------------------------------
bbox = [42.6768, 17.48022, 42.69809, 17.49266]  # [min_lon, min_lat, max_lon, max_lat]
download_zoom = 18
image = "satellite.tif"
mask_path = "mask.tif"
clean_raster_path = "segments_clean.tif"
vector_path = "segments.geojson"
vector_regularized_path = "segments_regularized.geojson"
model_id = "sam2-hiera-large"
min_object_size = 200  # pixels (increase to remove more small blobs)
do_regularize = True   # regularize polygons (good for buildings)

# Optional: supply a GeoJSON/CSV path (in Colab Files) with point prompts
# Examples: geojson_or_csv = "/content/my_points.geojson"  or "/content/points.csv"
geojson_or_csv = ""  # leave empty to use map clicks or default 4 points

# Your 4 points were originally given as (lat, lon). We'll store them that way:
user_points_latlon = [
    [17.483809, 42.686230],
    [17.483830, 42.686198],
    [17.484203, 42.688708],
    [17.484444, 42.685768],
]

def center_of_bbox(b):
    minx, miny, maxx, maxy = b
    return [(miny + maxy) / 2.0, (minx + maxx) / 2.0]  # [lat, lon]

center_lat, center_lon = center_of_bbox(bbox)
aoi_poly = Polygon([(bbox[0], bbox[1]), (bbox[2], bbox[1]), (bbox[2], bbox[3]), (bbox[0], bbox[3])])
aoi_box = box(*bbox)

## 1) Pick points: draw on the map (or provide a GeoJSON/CSV path above)

In [ ]:
m = leafmap.Map(center=[center_lat, center_lon], zoom=16, height="600px")
m.add_basemap("Satellite")
m.add_gdf(leafmap.gdf_from_geometries([aoi_poly], crs="EPSG:4326"), layer_name="AOI (BBox)")
m.add_draw_control(marker=True, circle=False, polygon=False, polyline=False, rectangle=False)
m.add_layers_control()
m

## 2) Download imagery for the bbox

In [ ]:
leafmap.map_tiles_to_geotiff(
    output=image,
    bbox=bbox,                 # [min_lon, min_lat, max_lon, max_lat]
    zoom=download_zoom,
    source="Satellite",
    overwrite=True,
)
assert os.path.exists(image), f"Image not found: {image}"
print("Saved:", image)

## 3) Normalize point prompts (map / GeoJSON / CSV) and keep only those inside the AOI\n- Accepts points in **lon,lat** or **lat,lon**; auto-swaps if needed\n- If nothing is provided/drawn, it will use your 4 points

In [ ]:
def read_points_from_csv(path):
    pts = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.lower().startswith("lon"):
                continue
            parts = [p.strip() for p in line.split(",")]
            if len(parts) >= 2:
                pts.append([float(parts[0]), float(parts[1])])
    return pts

def lonlat_in_bbox(lon, lat, b):
    return (b[0] <= lon <= b[2]) and (b[1] <= lat <= b[3])

def filter_points(points_lonlat, b):
    return [[x, y] for (x, y) in points_lonlat if lonlat_in_bbox(x, y, b)]

def swap_latlon(points):
    return [[y, x] for (x, y) in points]

def gdf_points_to_lonlat_list(gdf):
    rows = []
    for geom in gdf.geometry:
        if geom.geom_type == "Point":
            rows.append([float(geom.x), float(geom.y)])
        elif geom.geom_type == "MultiPoint":
            for pt in geom.geoms:
                rows.append([float(pt.x), float(pt.y)])
    return rows

# 1) Gather points from map, file, or your 4 defaults
points_lonlat = []

if geojson_or_csv and os.path.exists(geojson_or_csv):
    if geojson_or_csv.lower().endswith(".csv"):
        # Assume columns: lon,lat (if not, auto-swap below will handle)
        points_lonlat = read_points_from_csv(geojson_or_csv)
    else:
        gdf = gpd.read_file(geojson_or_csv)
        if gdf.crs is None:
            gdf = gdf.set_crs(epsg=4326)
        else:
            gdf = gdf.to_crs(epsg=4326)
        points_lonlat = gdf_points_to_lonlat_list(gdf)

elif getattr(m, "user_rois", None):
    # Leafmap returns [lon, lat]
    points_lonlat = [[float(x), float(y)] for [x, y] in m.user_rois]

else:
    # Use your 4 points (given as lat,lon) -> convert to lon,lat
    points_lonlat = [[lon, lat] for (lat, lon) in user_points_latlon]

print("Total points provided:", len(points_lonlat))

# 2) Filter to AOI. If none in AOI, try swapping (assume they were lat,lon)
in_aoi = filter_points(points_lonlat, bbox)
if len(in_aoi) == 0:
    swapped = swap_latlon(points_lonlat)  # interpret input as (lat,lon)
    in_aoi = filter_points(swapped, bbox)
    if len(in_aoi) > 0:
        points_lonlat = in_aoi  # use swapped interpretation
    else:
        raise SystemExit("No points fall inside the AOI. Check bbox or point coordinates/order/CRS.")
else:
    points_lonlat = in_aoi

print("Points inside AOI:", len(points_lonlat))
points_lonlat[:5]

## 4) Run SAM2 (set image ➜ predict_by_points)

In [ ]:
sam = SamGeo2(model_id=model_id, automatic=False)
sam.set_image(image)
assert hasattr(sam, "image") and sam.image is not None, "sam.set_image(image) did not run"

sam.predict_by_points(
    point_coords_batch=points_lonlat,   # list of [lon, lat]
    point_crs="EPSG:4326",
    output=mask_path,
    dtype="uint8",
    multimask_output=False,
)
print("Saved:", mask_path)

## 5) Preview mask on a map

In [ ]:
m2 = leafmap.Map(center=[center_lat, center_lon], zoom=17, height="600px")
m2.add_basemap("Satellite")
m2.add_raster(mask_path, cmap="viridis", nodata=0, opacity=0.6, layer_name="Mask")
m2.add_gdf(leafmap.gdf_from_geometries([aoi_poly], crs="EPSG:4326"), layer_name="AOI (BBox)")
m2

## 6) Clean small blobs & vectorize (GeoJSON) + optional regularization

In [ ]:
_arr, _gdf = sam.region_groups(
    mask_path,
    min_size=min_object_size,
    out_vector=vector_path,
    out_image=clean_raster_path,
)
print("Saved:", clean_raster_path)
print("Saved:", vector_path)

if do_regularize:
    try:
        regularize(vector_path, vector_regularized_path)
        print("Saved:", vector_regularized_path)
    except Exception as e:
        print("Regularization skipped/failed:", e)

## 7) Preview vectors

In [ ]:
m3 = leafmap.Map(center=[center_lat, center_lon], zoom=17, height="600px")
m3.add_basemap("Satellite")
if os.path.exists(vector_path):
    m3.add_vector(vector_path, layer_name="Segments (raw)")
if do_regularize and os.path.exists(vector_regularized_path):
    m3.add_vector(vector_regularized_path, layer_name="Segments (regularized)")
m3.add_gdf(leafmap.gdf_from_geometries([aoi_poly], crs="EPSG:4326"), layer_name="AOI (BBox)")
m3